In [1]:
import re
import numpy as np
import pandas as pd

In [2]:
def extract_geolocation_id(file_path):
    try:
        if pd.isna(file_path):
            return 'Unknown'
        file_name = file_path.split('/')[-1]  
        match = re.search(r'(\d{5}_\d+)', file_name)
        if match:
            return match.group(1)
        else:
            return 'Unknown'
    except Exception as e:
        print(f"Error processing file path '{file_path}': {e}")
        return 'Unknown'

roman_to_numeric = {
    'I': 1, 'II': 2, 'III': 3, 'IV': 4, 'V': 5, 'VI': 6,
    'VII': 7, 'VIII': 8, 'IX': 9, 'X': 10, 'XI': 11, 'XII': 12
}

def convert_mmi_to_numeric(mmi_value):
    if pd.isna(mmi_value):
        return np.nan
    if isinstance(mmi_value, (int, float)):
        return float(mmi_value)
    
    mmi_str = str(mmi_value).strip().upper()
    
    if mmi_str in roman_to_numeric:
        return float(roman_to_numeric[mmi_str])
    
    for roman, numeric in roman_to_numeric.items():
        if roman in mmi_str:
            return float(numeric)
    
    return np.nan

In [15]:
prompt_df = pd.read_csv('2019_ridgecrest_samples_prompt.csv')
prompt_df = prompt_df[['location_id', 'City']]

In [18]:
# model_files = [
#     'result_zipcode/2014_napa_B+G+B+C+V_gpt-4.1-mini-2025-04-14.json',
#     'result_zipcode/2014_napa_B+G+B+C+V_claude-3-5-haiku-20241022.json',
#     'result_zipcode/2014_napa_B+G+B+C+V_gpt-4o-2024-08-06.json',
#     'result_zipcode/2014_napa_B+G+B+C+V_Qwen2.5-VL-32B-Instruct.json',
#     'result_zipcode/2014_napa_B+G+B+C+V_Qwen2.5-VL-72B-Instruct.json',
#     'result_zipcode/2014_napa_B+G+B+C+V_Llama-3.2-90B-Vision-Instruct-Turbo.json'
# ]

model_files = [
    'result_zipcode/2019_ridgecrest_B+G+B+C+V_gpt-4.1-mini-2025-04-14.json',
    'result_zipcode/2019_ridgecrest_B+G+B+C+V_claude-3-5-haiku-20241022.json',
    'result_zipcode/2019_ridgecrest_B+G+B+C+V_gpt-4o-2024-08-06.json',
    'result_zipcode/2019_ridgecrest_B+G+B+C+V_Qwen2.5-VL-32B-Instruct.json',
    'result_zipcode/2019_ridgecrest_B+G+B+C+V_Qwen2.5-VL-72B-Instruct.json',
    'result_zipcode/2019_ridgecrest_B+G+B+C+V_Llama-3.2-90B-Vision-Instruct-Turbo.json'
]

model_names = [
    'gpt-4.1-mini',
    'claude-3.5-haiku',
    'gpt-4o',
    'qwen2.5-vl-32b',
    'qwen2.5-vl-72b',
    'llama-3.2-90b'
]

all_dfs = []

for file_path, model_name in zip(model_files, model_names):
    try:
        df = pd.read_json(file_path)
        df['location_id'] = df['file_path'].apply(extract_geolocation_id)
        df = pd.merge(df, prompt_df, on='location_id', how='left')
        
        df['MMI_numeric'] = df['MMI'].apply(convert_mmi_to_numeric)
        df['MMI_predicted_numeric'] = df['MMI_predicted'].apply(convert_mmi_to_numeric)
        
        df['model'] = model_name
        all_dfs.append(df[['City', 'MMI_numeric', 'MMI_predicted_numeric', 'model']])
        print(f"Loaded {model_name}: {len(df)} records")
    except Exception as e:
        print(f"Failed to load {model_name}: {e}")

combined_df = pd.concat(all_dfs, ignore_index=True)

pivot_predicted = combined_df.pivot_table(
    index='City', 
    columns='model', 
    values='MMI_predicted_numeric', 
    aggfunc='mean'
).round(2)

city_stats = combined_df.groupby('City').agg({
    'MMI_numeric': ['mean', 'count']
}).round(2)

city_stats.columns = ['Average_MMI', 'Count']
city_stats = city_stats.reset_index()
city_stats['Count'] = (city_stats['Count'] / len([df for df in all_dfs if len(df) > 0])).astype(int)

result = pd.merge(city_stats, pivot_predicted.reset_index(), on='City', how='left')

available_models = [col for col in pivot_predicted.columns if col in result.columns]
column_order = ['City', 'Count'] + available_models + ['Average_MMI']
result = result[column_order]

result = result.sort_values('Count', ascending=False)
result = result.reset_index()
result.head(50)

Loaded gpt-4.1-mini: 5000 records
Loaded claude-3.5-haiku: 5000 records
Loaded gpt-4o: 5000 records
Loaded qwen2.5-vl-32b: 5000 records
Loaded qwen2.5-vl-72b: 5000 records
Loaded llama-3.2-90b: 5000 records


,index,City,Count,claude-3.5-haiku,gpt-4.1-mini,gpt-4o,llama-3.2-90b,qwen2.5-vl-32b,qwen2.5-vl-72b,Average_MMI
0,30,Los Angeles,500,4.80,5.49,5.61,6.00,4.58,5.43,4.00
1,28,Las Vegas,300,4.25,4.36,4.93,5.96,4.62,5.09,4.00
2,4,Bakersfield,250,4.93,5.47,5.96,6.02,5.33,5.15,4.00
3,52,San Diego,200,4.26,3.87,4.11,5.87,3.58,3.78,3.00
4,27,Lancaster,150,4.73,5.49,6.22,6.11,5.00,5.28,4.67
5,22,Huntington Beach,100,4.57,4.36,5.62,6.03,3.96,4.56,4.00
6,49,Redondo Beach,100,4.46,4.77,5.95,5.98,4.06,4.81,4.00
7,56,Simi Valley,100,4.68,4.70,5.50,6.01,4.31,5.51,4.00
8,20,Henderson,100,3.90,4.32,4.82,5.85,4.25,4.69,3.50
9,33,Mission Viejo,100,3.96,4.31,5.88,5.82,4.05,4.50,4.00
